# Labs 19 and 20: Comps, calculators and chart reads

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

In [ ]:
# From the slide: Calculators that leave a trace
import hashlib, json
def trace(name, value, inputs, formula):
    t = dict(calc=name, value=value, inputs=inputs,
             formula=formula)
    raw = json.dumps(t, sort_keys=True).encode()
    t["calc_id"] = hashlib.sha1(raw).hexdigest()[:12]
    return t

def ltm(fy, cur, prior):             # three facts
    v = fy["value"] + cur["value"] - prior["value"]
    ids = [f["fact_id"] for f in (fy, cur, prior)]
    return trace("ltm", v, ids, "FY + YTD_cur - YTD_prior")

def ev(price, sh, debt, cash, pref=0.0, nci=0.0):
    v = (price * sh["value"] + debt["value"]
         + pref + nci - cash["value"])
    ids = [f["fact_id"] for f in (sh, debt, cash)]
    return trace("ev", v, ids, "P*sh+debt+pref+nci-cash")

In [ ]:
import pandas as pd
cf = pd.read_csv("data/reference/comps_facts_sample.csv"); px = pd.read_csv("data/reference/prices_asof_20260930.csv").set_index("ticker")
def fact(t, name, v): return {"fact_id": f"F-{t}-{name}", "value": float(v)}
rows = []
for _, r in cf.iterrows():
    t = r["ticker"]
    rev = ltm(fact(t, "fy_rev", r.fy_revenue), fact(t, "ytd_rev", r.ytd_revenue_current), fact(t, "ytd_rev_prior", r.ytd_revenue_prior))
    ebitda = ltm(fact(t, "fy_ebitda", r.fy_ebitda), fact(t, "ytd_ebitda", r.ytd_ebitda_current), fact(t, "ytd_ebitda_prior", r.ytd_ebitda_prior))
    e = ev(px.loc[t, "close"], fact(t, "shares", r.shares_out), fact(t, "debt", r.debt), fact(t, "cash", r.cash), pref=r.preferred, nci=r.nci)
    evr = trace("ev_revenue", e["value"] / rev["value"], [e["calc_id"], rev["calc_id"]], "EV / LTM revenue")
    eve = trace("ev_ebitda", e["value"] / ebitda["value"], [e["calc_id"], ebitda["calc_id"]], "EV / LTM EBITDA") if ebitda["value"] > 0 else None
    rows.append({"ticker": t, "ltm_revenue": rev["value"], "ltm_revenue_calc_id": rev["calc_id"], "ev": e["value"], "ev_calc_id": e["calc_id"],
                 "ev_revenue": evr["value"], "ev_revenue_calc_id": evr["calc_id"],
                 "ev_ebitda": eve["value"] if eve else None, "ev_ebitda_calc_id": eve["calc_id"] if eve else None})   # n.m. when EBITDA <= 0
comps = pd.DataFrame(rows)
Path("outputs/comps").mkdir(parents=True, exist_ok=True); comps.to_csv("outputs/comps/deal_a.csv", index=False)   # CS2 section 8: a calc_id beside every metric
print(comps[["ticker", "ev_revenue", "ev_revenue_calc_id", "ev_ebitda", "ev_ebitda_calc_id"]].round(2).to_string(index=False))
print(comps[["ev_revenue", "ev_ebitda"]].quantile([.25, .5, .75]).round(2))   # n.m. (negative EBITDA) excluded

In [ ]:
# Chart vs table: a recorded chart read against the seller's table
reads = json.load(open("data/dataroom/deal_a/chart_reads.json"))
F = {(f["metric"], f["period"]): f["value"] for f in sd.seller_facts()}
for c in reads:
    gap = (c["value"] - F[(c["metric"], c["period"])]) / F[(c["metric"], c["period"])]
    print(c["chart"], f"chart {c['value']/1e6:.1f}m vs table {F[(c['metric'], c['period'])]/1e6:.1f}m, gap {gap:.1%}",
          "-> DISCREPANCY" if abs(gap) > llm.P["reconcile"]["tolerance_rel"] else "-> ok")

**Exercise 19/20.** Trace SMPA's EV/EBITDA back to its input facts with the calc ids. Why is SMPD's EV/EBITDA left out of the quartiles?